# Week 3 Exercise - Synthetic Technical Interview Dataset Generator

A Gradio-powered tool that generates synthetic technical interview Q&A datasets.

## Features
- Interactive UI to select category, difficulty, and number of samples
- Multiple categories: Python, Data Structures, Algorithms, ML, System Design, LLM Engineering
- LLM-powered generation for realistic questions and answers
- Export to JSON format

## Use Cases
- Training data for interview prep chatbots
- Evaluation datasets for technical assessments
- Study material generation

In [ ]:
# Imports

import os
import re
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Setup

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# Setup client (works with OpenRouter or direct OpenAI)
if openai_api_key and openai_api_key.startswith('sk-or-'):
    # OpenRouter
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("Using OpenRouter")
else:
    # Direct OpenAI
    client = OpenAI()
    print("Using OpenAI directly")

# Available models
MODELS = {
    "GPT-4.1-mini": "gpt-4.1-mini",
    "GPT-4.1": "gpt-4.1",
    "Claude Sonnet": "anthropic/claude-sonnet-4",
}
DEFAULT_MODEL = "GPT-4.1-mini"

In [ ]:
# Interview Categories with schema hints

CATEGORIES = {
    "Python Fundamentals": {
        "description": "Core Python language features, syntax, and best practices",
        "schema": "Each object: question, answer (detailed explanation), code_example (if applicable), follow_up_questions (array of 2 strings), difficulty, tags (array)",
        "topics": ["decorators", "generators", "context managers", "GIL", "memory management", "metaclasses", "descriptors", "type hints"]
    },
    "Data Structures": {
        "description": "Data structures, implementations, and complexity analysis",
        "schema": "Each object: question, answer, code_example, time_complexity, space_complexity, follow_up_questions, difficulty, tags",
        "topics": ["hash tables", "binary trees", "heaps", "graphs", "linked lists", "tries", "stacks", "queues"]
    },
    "Algorithms": {
        "description": "Algorithmic problem-solving, optimization, and complexity",
        "schema": "Each object: question, answer, code_example, time_complexity, approach (e.g. divide-and-conquer, DP), follow_up_questions, difficulty, tags",
        "topics": ["sorting", "searching", "dynamic programming", "greedy", "graph algorithms", "recursion", "backtracking"]
    },
    "Machine Learning": {
        "description": "ML concepts, models, evaluation, and practical implementation",
        "schema": "Each object: question, answer, code_example (sklearn/pytorch if applicable), key_concepts (array), follow_up_questions, difficulty, tags",
        "topics": ["supervised learning", "unsupervised learning", "neural networks", "regularization", "cross-validation", "feature engineering", "model evaluation"]
    },
    "System Design": {
        "description": "System architecture, scalability, and distributed systems",
        "schema": "Each object: question, answer, key_components (array), trade_offs, follow_up_questions, difficulty, tags",
        "topics": ["load balancing", "caching", "databases", "message queues", "microservices", "API design", "scalability"]
    },
    "LLM Engineering": {
        "description": "LLM APIs, prompt engineering, RAG, and AI application development",
        "schema": "Each object: question, answer, code_example (API usage if applicable), best_practices (array), follow_up_questions, difficulty, tags",
        "topics": ["prompt engineering", "RAG", "fine-tuning", "embeddings", "tokenization", "agents", "function calling", "context management"]
    }
}

DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard", "Mixed"]

print(f"Categories: {list(CATEGORIES.keys())}")

In [ ]:
# Prompt Generation

def get_system_prompt(category: str, difficulty: str, num_samples: int) -> str:
    """Generate system prompt for the LLM."""
    cat_info = CATEGORIES.get(category, {})
    schema = cat_info.get("schema", "Each object: question, answer, code_example, follow_up_questions, difficulty, tags")
    topics = cat_info.get("topics", [])
    
    difficulty_guidance = ""
    if difficulty == "Easy":
        difficulty_guidance = "Questions should be suitable for junior developers, focusing on fundamentals."
    elif difficulty == "Medium":
        difficulty_guidance = "Questions should be suitable for mid-level developers, requiring practical experience."
    elif difficulty == "Hard":
        difficulty_guidance = "Questions should be suitable for senior developers, requiring deep expertise."
    else:  # Mixed
        difficulty_guidance = "Include a mix of easy, medium, and hard questions."
    
    return f"""You are a technical interview expert. Generate exactly {num_samples} realistic technical interview questions and comprehensive answers.

Category: {category}
Description: {cat_info.get('description', category)}
Relevant topics: {', '.join(topics)}

Difficulty: {difficulty}
{difficulty_guidance}

Schema (use these field names): {schema}

Requirements:
- Questions should be realistic interview questions asked at top tech companies
- Answers should be thorough, accurate, and demonstrate expertise
- Include practical code examples where applicable
- Follow-up questions should probe deeper understanding

Output valid JSON only: a single object with key "items" containing an array of {num_samples} objects.
No markdown code fences, no explanations. Just the JSON object.

Example structure:
{{
  "items": [
    {{
      "question": "...",
      "answer": "...",
      "code_example": "...",
      "follow_up_questions": ["...", "..."],
      "difficulty": "medium",
      "tags": ["...", "..."]
    }}
  ]
}}"""

In [ ]:
# JSON Extraction Helper

def extract_json_array(text: str) -> list:
    """Extract JSON array from model output."""
    if not text or not text.strip():
        raise ValueError("Empty model output.")
    
    t = text.strip()
    
    # Remove markdown code fences if present
    if "```" in t:
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE | re.MULTILINE)
        t = re.sub(r"\s*```$", "", t, flags=re.MULTILINE)
        t = t.strip()
    
    # Try to parse directly
    try:
        obj = json.loads(t)
    except json.JSONDecodeError:
        # Try to find JSON object
        start = t.find("{")
        end = t.rfind("}")
        if start != -1 and end != -1 and end > start:
            t = t[start:end+1]
            # Fix trailing commas
            t = re.sub(r",\s*([\]}])", r"\1", t)
            obj = json.loads(t)
        else:
            raise ValueError("No valid JSON found in output.")
    
    # Extract array from object
    if isinstance(obj, list):
        return obj
    
    for key in ("items", "data", "entries", "results", "questions"):
        if isinstance(obj.get(key), list):
            return obj[key]
    
    raise ValueError("JSON object has no recognizable array field.")

In [ ]:
# Main Generator Function

def generate_interview_data(
    category: str,
    difficulty: str,
    num_samples: int,
    additional_context: str,
    model_name: str
) -> str:
    """Generate synthetic interview Q&A data."""
    
    if not category:
        return json.dumps({"error": "Please select a category."}, indent=2)
    
    num_samples = max(1, min(15, int(num_samples)))
    
    system_prompt = get_system_prompt(category, difficulty, num_samples)
    
    user_prompt = additional_context.strip() if additional_context else f"Generate {num_samples} high-quality {category} interview questions."
    
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    
    try:
        print(f"Generating {num_samples} {difficulty} {category} questions using {model_name}...")
        
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=4000
        )
        
        raw_content = response.choices[0].message.content or ""
        
        # Extract and format JSON
        items = extract_json_array(raw_content)
        
        result = {
            "metadata": {
                "category": category,
                "difficulty": difficulty,
                "count": len(items),
                "model": model_name
            },
            "items": items
        }
        
        print(f"Successfully generated {len(items)} items.")
        return json.dumps(result, indent=2)
        
    except Exception as e:
        error_result = {
            "error": str(e),
            "hint": "Check your API key and try again."
        }
        return json.dumps(error_result, indent=2)

In [ ]:
# Gradio UI

with gr.Blocks(title="Technical Interview Data Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # Technical Interview Dataset Generator
    
    Generate synthetic technical interview Q&A data for training, evaluation, or study purposes.
    
    **How to use:**
    1. Select a category (Python, Data Structures, ML, etc.)
    2. Choose difficulty level
    3. Set number of samples (1-15)
    4. Optionally add specific topics or context
    5. Click Generate
    """)
    
    with gr.Row():
        with gr.Column():
            category_dropdown = gr.Dropdown(
                choices=list(CATEGORIES.keys()),
                value=list(CATEGORIES.keys())[0],
                label="Category"
            )
            
            difficulty_dropdown = gr.Dropdown(
                choices=DIFFICULTY_LEVELS,
                value="Medium",
                label="Difficulty"
            )
        
        with gr.Column():
            num_samples_slider = gr.Slider(
                minimum=1,
                maximum=15,
                value=5,
                step=1,
                label="Number of Questions"
            )
            
            model_dropdown = gr.Dropdown(
                choices=list(MODELS.keys()),
                value=DEFAULT_MODEL,
                label="Model"
            )
    
    context_input = gr.Textbox(
        label="Additional Context (optional)",
        placeholder="e.g., Focus on async/await patterns, or Include questions about PyTorch",
        lines=2
    )
    
    generate_btn = gr.Button("Generate Dataset", variant="primary")
    
    output_json = gr.Code(
        label="Generated Dataset (JSON)",
        language="json",
        lines=25
    )
    
    # Examples
    gr.Markdown("### Examples")
    gr.Examples(
        examples=[
            ["Python Fundamentals", "Medium", 5, "Focus on decorators and context managers", "GPT-4.1-mini"],
            ["Machine Learning", "Hard", 3, "Deep learning and neural network architecture questions", "GPT-4.1-mini"],
            ["System Design", "Hard", 3, "Distributed systems and scalability", "GPT-4.1-mini"],
            ["LLM Engineering", "Medium", 5, "RAG and prompt engineering best practices", "GPT-4.1-mini"],
        ],
        inputs=[category_dropdown, difficulty_dropdown, num_samples_slider, context_input, model_dropdown]
    )
    
    # Connect button to function
    generate_btn.click(
        fn=generate_interview_data,
        inputs=[category_dropdown, difficulty_dropdown, num_samples_slider, context_input, model_dropdown],
        outputs=output_json
    )

In [ ]:
# Launch the app
app.launch()

## Usage Notes

### Output Format
The generated JSON includes:
- **metadata**: Category, difficulty, count, model used
- **items**: Array of Q&A objects with:
  - question: The interview question
  - answer: Comprehensive answer
  - code_example: Practical code (where applicable)
  - follow_up_questions: Array of deeper questions
  - difficulty: easy/medium/hard
  - tags: Relevant topic tags

### Saving Data
Copy the JSON output and save to a file, or use programmatically:
```python
import json
data = json.loads(output_string)
with open('interview_data.json', 'w') as f:
    json.dump(data, f, indent=2)
```

### Categories Available
- **Python Fundamentals**: Decorators, generators, GIL, memory management
- **Data Structures**: Trees, graphs, hash tables, complexity analysis
- **Algorithms**: Sorting, DP, graph algorithms, optimization
- **Machine Learning**: Models, evaluation, feature engineering
- **System Design**: Scalability, distributed systems, architecture
- **LLM Engineering**: Prompts, RAG, embeddings, agents